In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# ==========================================
# 1. THE MULTIMODAL ARCHITECTURE
# ==========================================
class MultimodalSepsisNet(nn.Module):
    def __init__(self, clinical_input_dim, genomic_input_dim, latent_dim=64):
        super(MultimodalSepsisNet, self).__init__()
        
        # Branch 1: The ICU Doctor (Clinical Encoder)
        self.clinical_encoder = nn.Sequential(
            nn.Linear(clinical_input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, latent_dim),
            nn.ReLU()
        )
        
        # Branch 2: The Geneticist (Genomic Encoder)
        self.genomic_encoder = nn.Sequential(
            nn.Linear(genomic_input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, latent_dim),
            nn.ReLU()
        )
        
        # The Fusion Center (Combining both latent spaces)
        # Latent_dim * 2 because we concatenate them
        self.fusion_classifier = nn.Sequential(
            nn.Linear(latent_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Sigmoid() # Outputs a probability between 0 and 1
        )

    def forward(self, clinical_x, genomic_x):
        # 1. Compress both modalities into the latent space
        clinical_latent = self.clinical_encoder(clinical_x)
        genomic_latent = self.genomic_encoder(genomic_x)
        
        # 2. Fuse them together (concatenate along the feature dimension)
        fused_latent = torch.cat((clinical_latent, genomic_latent), dim=1)
        
        # 3. Make the final prediction
        output = self.fusion_classifier(fused_latent)
        return output

# ==========================================
# 2. TESTING THE PIPES (DUMMY DATA)
# ==========================================
print("[*] Generating Synthetic Paired Dataset for Architecture Test...")
num_samples = 1000
clin_dim = 115   # Matches your PhysioNet output
gen_dim = 501    # Matches your GEO PCA output

# Create fake tensors that mimic our exact data shapes
dummy_clinical = torch.randn(num_samples, clin_dim)
dummy_genomic = torch.randn(num_samples, gen_dim)
dummy_labels = torch.randint(0, 2, (num_samples, 1)).float()

# Load into PyTorch DataLoader
dataset = TensorDataset(dummy_clinical, dummy_genomic, dummy_labels)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# ==========================================
# 3. INITIALIZE AND TEST THE MODEL
# ==========================================
print("[*] Initializing MultimodalSepsisNet...")
model = MultimodalSepsisNet(clinical_input_dim=clin_dim, genomic_input_dim=gen_dim)

criterion = nn.BCELoss() # Binary Cross Entropy for Yes/No prediction
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("[*] Running a 3-Epoch Test to verify gradient flow...")
for epoch in range(3):
    epoch_loss = 0.0
    for clin_batch, gen_batch, label_batch in dataloader:
        optimizer.zero_grad()
        
        # Forward pass through both branches simultaneously
        predictions = model(clin_batch, gen_batch)
        
        # Calculate loss and update weights
        loss = criterion(predictions, label_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"   -> Epoch {epoch+1}/3 | Loss: {epoch_loss/len(dataloader):.4f}")

print("\n[+] SUCCESS: Architecture compiled, latent space mapped, and fusion successful.")
print(f"[+] Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

[*] Generating Synthetic Paired Dataset for Architecture Test...
[*] Initializing MultimodalSepsisNet...
[*] Running a 3-Epoch Test to verify gradient flow...
   -> Epoch 1/3 | Loss: 0.6946
   -> Epoch 2/3 | Loss: 0.6058
   -> Epoch 3/3 | Loss: 0.3806

[+] SUCCESS: Architecture compiled, latent space mapped, and fusion successful.
[+] Total trainable parameters: 177,153
